# R_eff,ij Onion-Layer Curves vs (N, a, d): does curve shape help invert the network?

`inverse_N_a_from_R_eff.ipynb` inverts the *scalar* source-to-drain R_eff over (N, a) only --
it never touches d -- and even then the inversion is fundamentally underdetermined: R_eff
collapses onto the combined variable ~N*a^2 (more, shorter sticks behave like fewer, longer
ones), so a target R_eff pins down a *curve* of (N, a) solutions, not a unique pair. With d as a
genuine third free parameter, a single scalar R_eff is even less informative.

This notebook asks whether the **onion-layer protocol** (`utils/multimeter.py`:
R_eff(reference_node, j) swept outward through concentric distance shells, see
`multimeter_r_eff.ipynb`) gives enough *extra, independent* information to help: does the
*shape* of the mean R_eff,ij(r) curve -- not just its overall scale -- depend on N, a, d in
distinguishable ways?

**Pipeline:**
- Part A: a small local subset, computed live, to sanity-check the pipeline end-to-end and
  reconfirm the pinv fast path (`multimeter.run_onion_protocol_pinv`, see
  `multimeter_r_eff.ipynb` Section 6b) against the solve-based path on a real network here too.
- Part B: load the full (N, a, d) grid of mean onion curves from
  `cluster/onion_grid_results/chunk_*.json` (produced by `cluster/run_onion_grid_task.py` /
  `run_onion_grid_array.sbatch` on Apuana), paired by construction with the existing scalar
  R_eff grid in `cluster/effective_resistance_d_results/` (same seed formula, same networks --
  see `utils/pipeline_utils.py`'s `stable_seed()`).
- Part C: the actual diagnostic -- for (N, a, d) triples that land close together in scalar
  R_eff, do their mean onion curves diverge in shape? This is the direct answer to the question
  above, and sets up whether a follow-on curve-based (N, a, d) inversion notebook (the 3-parameter
  analogue of `inverse_N_a_from_R_eff.ipynb`) is worth building.

This notebook deliberately stops at the diagnostic -- it does not itself fit an inversion model.

---
## 1. Setup and imports

In [1]:
import sys
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    utils_path = "/content/drive/MyDrive/ColabProject/utils"
    cluster_dir = "/content/drive/MyDrive/ColabProject/cluster"
    print("Environment: Google Colab")
except ImportError:
    utils_path = os.path.abspath(os.path.join(os.getcwd(), "..", "utils"))
    cluster_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "cluster"))
    print("Environment: Local")

if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Utils path: {utils_path}")
print(f"Cluster dir: {cluster_dir}")

Environment: Local
Utils path: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils
Cluster dir: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/cluster


In [2]:
from pipeline_utils_3d import generate_percolating_network_3d
from pipeline_utils import distance_dependent_impedance, stable_seed
from multimeter import run_onion_protocol, run_onion_protocol_pinv
from experiment_manager import setup_experiment

exp_manager = setup_experiment()

EXPERIMENT CONFIGURATION
  Environment: Local
  Experiment #: 877
  Timestamp: 20260915_205825
  Results directory: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_877



---
# Part A -- Local validation subset

A handful of (N, a, d) combos, computed live and cheaply (small N, small a -- the same corner
`run_onion_grid_task.py`'s chunk 0 covers), to sanity-check the pipeline end-to-end before
trusting Part B/C's cluster-scale data. `R0`, `CONTACT_DISTANCE`, and the seed formula match
`cluster/run_onion_grid_task.py` exactly.

In [3]:
R0 = 1000.0
CONTACT_DISTANCE = 0.03
MAX_ATTEMPTS = 500
N_TRIALS_PREVIEW = 50   # error bars on the preview figure are meaningless at 4
DRAW_BUDGET = 40000     # per-combo draw cap; see run_one_combo_preview
LAYER_DISTANCES = [round(0.1 * n, 2) for n in range(1, 10)]

# Generator boundary rule (utils/point_generator.py): 'clip' since 2026-09-15. The preview
# combos were (70, 0.2, 0.1), (70, 0.2, 0.15), (45, 0.2, 0.1) under the old 'reject' rule;
# they no longer percolate at usable rates under 'clip', so the same two contrasts are
# taken at higher N (rates 32%, 12%, 10%). Must match cluster/run_preview_curves_task.py.
BOUNDARY = "clip"
preview_combos = [(150, 0.2, 0.1), (150, 0.2, 0.15), (125, 0.2, 0.1)]
# Part D (boosted rerun) is not part of the clipping-rule round: the base grid now has
# 24 realizations per combination. Set True only when onion_grid_boost_results/ holds a
# boost run made with the same rule and plan.
RUN_BOOST = False
print(f"Part A preview combos: {preview_combos}")

Part A preview combos: [(150, 0.2, 0.1), (150, 0.2, 0.15), (125, 0.2, 0.1)]


In [4]:
def run_one_combo_preview(N, a, d, n_trials):
    """Same logic as cluster/run_onion_grid_task.py's run_one_combo, inline here for a
    quick local look (not meant to replace the cluster script for the real grid)."""
    seed_base = stable_seed(N, round(a, 3), round(d, 3), 'ext')
    np.random.seed(seed_base)

    R_effs = []
    per_network_layer_means = []
    total_attempts = 0
    sample_res = None

    # NOTE: the grid scripts stop at the first 500-attempt block that fails to
    # percolate, which cannot reach n_trials=50 at these rates (the (45, 0.2, 0.1)
    # corner percolates on ~1 draw in 3000). Here we keep drawing to a global budget
    # instead. Draws are i.i.d., so this only lengthens the sample, it does not bias it.
    while len(R_effs) < n_trials and total_attempts < DRAW_BUDGET:
        res, attempts = generate_percolating_network_3d(
            N=N, a=a, contact_distance=CONTACT_DISTANCE,
            box_dims=(1.0, 1.0, d), k_uniform=R0, max_attempts=MAX_ATTEMPTS, boundary=BOUNDARY
        )
        total_attempts += attempts
        if res is None:
            continue
        if sample_res is None:
            sample_res = res

        rl = distance_dependent_impedance(res, R0)
        R_effs.append(rl['impedance'])

        reference_node = res['pre_merge_source_nodes'][0]
        records = run_onion_protocol_pinv(
            res['pre_merge_intersections'], res['pre_merge_segment_to_nodes'],
            reference_node, layer_distances=LAYER_DISTANCES, R0=R0
        )
        layer_means = {}
        for layer_d in LAYER_DISTANCES:
            vals = [r['r_eff'] for r in records if r['layer_distance'] == layer_d and np.isfinite(r['r_eff'])]
            if vals:
                layer_means[layer_d] = float(np.mean(vals))
        per_network_layer_means.append(layer_means)

    onion_curve = {}
    for layer_d in LAYER_DISTANCES:
        per_real = [lm[layer_d] for lm in per_network_layer_means if layer_d in lm]
        if per_real:
            n_r = len(per_real)
            sd = float(np.std(per_real, ddof=1)) if n_r > 1 else 0.0
            onion_curve[layer_d] = {'mean': float(np.mean(per_real)), 'std': sd,
                                    'sem': sd / np.sqrt(n_r), 'n': n_r}

    return {
        'N': N, 'a': a, 'd': d, 'n': len(R_effs),
        'mean_ohm': float(np.mean(R_effs)) if R_effs else None,
        'R_eff_values': R_effs,
        'onion_curve': onion_curve,
        'sample_network': sample_res,
    }

preview_rows = []
for (N, a, d) in preview_combos:
    row = run_one_combo_preview(N, a, d, N_TRIALS_PREVIEW)
    preview_rows.append(row)
    mean_str = f"{row['mean_ohm']:.1f}" if row['mean_ohm'] is not None else "n/a"
    print(f"N={N:3d} a={a:.1f} d={d:.2f}  {row['n']}/{N_TRIALS_PREVIEW} percolating  "
          f"mean={mean_str}  onion layers covered={len(row['onion_curve'])}/{len(LAYER_DISTANCES)}")

N=150 a=0.2 d=0.10  50/50 percolating  mean=853.1  onion layers covered=9/9


N=150 a=0.2 d=0.15  50/50 percolating  mean=1126.1  onion layers covered=9/9


N=125 a=0.2 d=0.10  50/50 percolating  mean=1029.8  onion layers covered=9/9


### A.1 Re-confirm the pinv fast path against the solve-based path

Same check as `multimeter_r_eff.ipynb` Section 6b, on one of the networks generated above --
`run_onion_protocol` (one dense solve per probe) vs. `run_onion_protocol_pinv` (one pseudoinverse
per network) should agree to numerical precision.

In [5]:
def build_graph_from_segments(n_nodes, segment_to_nodes):
    g = [[] for _ in range(n_nodes)]
    for e, (n1, n2) in segment_to_nodes.items():
        g[n1].append(e)
        g[n2].append(e)
    return g

check_row = next(r for r in preview_rows if r['sample_network'] is not None)
res = check_row['sample_network']
inter = res['pre_merge_intersections']
seg = res['pre_merge_segment_to_nodes']
graph = build_graph_from_segments(len(inter), seg)
ref = res['pre_merge_source_nodes'][0]

slow_records = run_onion_protocol(
    inter, graph, seg, ref, layer_distances=LAYER_DISTANCES, k=R0,
    distance_dependent=True, max_probes_per_layer=None, seed=0, verbose=False
)
fast_records = run_onion_protocol_pinv(inter, seg, ref, layer_distances=LAYER_DISTANCES, R0=R0)

slow_by_j = {r['node_j']: r['r_eff'] for r in slow_records}
fast_by_j = {r['node_j']: r['r_eff'] for r in fast_records}
assert set(slow_by_j) == set(fast_by_j)

finite_js = [j for j in slow_by_j if np.isfinite(slow_by_j[j])]
rel_errors = [abs(fast_by_j[j] - slow_by_j[j]) / slow_by_j[j] for j in finite_js]
print(f"N={check_row['N']} a={check_row['a']} d={check_row['d']}: {len(finite_js)} finite probes compared")
print(f"Max relative error (fast vs. slow): {max(rel_errors):.2e}")
assert max(rel_errors) < 1e-6
print("PASSED")

N=150 a=0.2 d=0.1: 152 finite probes compared
Max relative error (fast vs. slow): 7.07e-13
PASSED


In [6]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for row in preview_rows:
    if not row['onion_curve']:
        continue
    ds = sorted(row['onion_curve'])
    means = [row['onion_curve'][d]['mean'] for d in ds]
    sems = [row['onion_curve'][d]['sem'] for d in ds]
    ax.errorbar(ds, means, yerr=sems, fmt='o-', capsize=3, markersize=4, linewidth=1.4,
                label=f"$N$={row['N']}, $a$={row['a']}, $d$={row['d']} ($n$={row['n']})")
ax.set_yscale('log')
ax.set_xlabel(r'Distance from reference node, $r$')
ax.set_ylabel(r'Mean $R_{\mathrm{eff},ij}$ per layer ($\Omega$)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()
exp_manager.save_figure(fig, 'partA_preview_onion_curves.png')


  [OK] Figure saved: partA_preview_onion_curves.png


/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_41168/3972107006.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'/Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_877/partA_preview_onion_curves.png'

---
# Part B -- Load the full (N, a, d) grid

Same 420-combo grid as `effective_resistance_vs_d.ipynb` Part C / `cluster/run_onion_grid_task.py`
-- `A_VALUES`/`D_VALUES`/`N_PER_A`/`N_FLOOR`/`NODE_CAP`/`N_TRIALS` below must match that script
exactly, since the cache-check keys results by (N, a, d) regardless of which chunk produced them.

In [7]:
A_VALUES = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1]
D_VALUES = [0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0]
N_PER_A = 6
N_FLOOR = 20
NODE_CAP = 1500
N_TASKS_ARRAY = 20

def n_max_for_a_d(a, d, node_cap=NODE_CAP, N_ceiling=150, N_floor=N_FLOOR, round_to=5):
    C, pN, pa, pd = 0.0016, 2.66, 2.05, -0.60
    raw = (node_cap / (C * a ** pa * d ** pd)) ** (1 / pN)
    capped = min(N_ceiling, raw)
    return int(max(N_floor, round(capped / round_to) * round_to))

grid_ext = {}
for a in A_VALUES:
    n_max = n_max_for_a_d(a, min(D_VALUES))
    Ns = sorted(set(int(round(x / 5) * 5) for x in np.linspace(N_FLOOR, n_max, N_PER_A)))
    grid_ext[a] = Ns

combos_ext = [(N, a, d) for a in A_VALUES for N in grid_ext[a] for d in D_VALUES]
print(f"{len(combos_ext)} (N, a, d) combinations in the full grid")

420 (N, a, d) combinations in the full grid


In [8]:
onion_dir = os.path.join(cluster_dir, "onion_grid_results")
scalar_dir = os.path.join(cluster_dir, "effective_resistance_d_results")

def load_chunks(results_dir, n_tasks=N_TASKS_ARRAY):
    chunk_files = [os.path.join(results_dir, f"chunk_{i:02d}.json") for i in range(n_tasks)]
    found = [p for p in chunk_files if os.path.exists(p)]
    rows = []
    for p in found:
        with open(p) as f:
            rows.extend(json.load(f))
    return rows, len(found), n_tasks

onion_rows, n_onion_found, n_tasks = load_chunks(onion_dir)
scalar_rows, n_scalar_found, _ = load_chunks(scalar_dir)

onion_by_key = {(r['N'], r['a'], r['d']): r for r in onion_rows}
scalar_by_key = {(r['N'], r['a'], r['d']): r for r in scalar_rows}

print(f"Onion grid:  {n_onion_found}/{n_tasks} chunk files found, {len(onion_by_key)} combos loaded")
print(f"Scalar grid: {n_scalar_found}/{n_tasks} chunk files found, {len(scalar_by_key)} combos loaded")
if n_onion_found < n_tasks:
    print("\n[INFO] Onion grid is incomplete -- run cluster/run_onion_grid_array.sbatch on Apuana "
          "for the full 420-combo dataset. Part C below works with whatever is currently loaded, "
          "which may be too sparse for a conclusive answer until that finishes.")

def _aligned_curves(row, shared_layers):
    """Individual realization curves at shared_layers, aligned by realization.
    Uses the per_realization_curves record (one dict per realization, None for
    empty shells) when the chunk has it; otherwise falls back to the per-shell
    lists, which are aligned only when every realization has every shell, so
    the fallback keeps only rows where that holds."""
    if 'per_realization_curves' in row:
        out = []
        for cv in row['per_realization_curves']:
            vals = [cv.get(str(d)) for d in shared_layers]
            if all(v is not None for v in vals):
                out.append(vals)
        return np.array(out, dtype=float)
    curve = row['onion_layer_curves']
    if any(curve[str(d)]['n_realizations'] != row['n'] for d in shared_layers if str(d) in curve):
        return np.zeros((0, len(shared_layers)))
    n_real = min(curve[str(d)]['n_realizations'] for d in shared_layers if str(d) in curve)
    return np.array([[curve[str(d)]['per_realization_means'][i] for d in shared_layers] for i in range(n_real)], dtype=float)



Onion grid:  20/20 chunk files found, 420 combos loaded
Scalar grid: 20/20 chunk files found, 420 combos loaded


---
# Part C -- Does onion-curve shape disambiguate (N, a, d)?

For (N, a, d) triples whose **scalar** R_eff lands close together (the kind of triples the
existing scalar inversion can't tell apart), compare their **mean onion curves**. If the curves
diverge in shape -- not just in overall amplitude -- the onion protocol carries real extra
information the scalar doesn't.

In [9]:
REL_TOL = 0.15  # group combos whose scalar mean R_eff are within 15% of each other

paired_keys = [k for k in onion_by_key if k in scalar_by_key and onion_by_key[k]['onion_layer_curves']
               and scalar_by_key[k]['mean_ohm'] is not None]
paired_keys.sort(key=lambda k: scalar_by_key[k]['mean_ohm'])
print(f"{len(paired_keys)} combos have both a scalar R_eff and a non-empty onion curve")

groups = []
used = set()
for i, k1 in enumerate(paired_keys):
    if k1 in used:
        continue
    r1 = scalar_by_key[k1]['mean_ohm']
    group = [k1]
    for k2 in paired_keys[i + 1:]:
        if k2 in used:
            continue
        r2 = scalar_by_key[k2]['mean_ohm']
        if abs(r2 - r1) / r1 <= REL_TOL and k2[:2] != k1[:2]:  # different (N, a), not just different d
            group.append(k2)
    if len(group) >= 2:
        groups.append(group)
        used.update(group)

print(f"{len(groups)} group(s) of >=2 (N, a, d) triples with scalar R_eff within {REL_TOL:.0%} of each other")
for g in groups[:10]:
    print(f"  {[(N, a, d, round(scalar_by_key[(N,a,d)]['mean_ohm'], 1)) for (N, a, d) in g]}")
if not groups:
    print("\n[INFO] No matched groups yet with the data currently loaded -- expected until the full "
          "grid (Part B) is available. Re-run this section after the Apuana job finishes.")

341 combos have both a scalar R_eff and a non-empty onion curve
32 group(s) of >=2 (N, a, d) triples with scalar R_eff within 15% of each other
  [(95, 1.1, 0.1, 26.8), (105, 1.0, 0.1, 27.5), (115, 0.9, 0.1, 28.4), (125, 0.8, 0.1, 29.3), (140, 0.7, 0.1, 29.9)]
  [(95, 1.1, 0.15, 31.1), (105, 1.0, 0.15, 31.2), (115, 0.9, 0.15, 32.6), (105, 1.0, 0.2, 34.0), (90, 1.0, 0.1, 34.6), (150, 0.6, 0.1, 35.1), (140, 0.7, 0.15, 35.3), (125, 0.8, 0.15, 35.4)]
  [(95, 1.1, 0.2, 33.9), (115, 0.9, 0.2, 36.4), (80, 1.1, 0.1, 36.8), (125, 0.8, 0.2, 36.9)]
  [(140, 0.7, 0.2, 39.1), (95, 0.9, 0.1, 39.4), (105, 0.8, 0.1, 40.5), (95, 1.1, 0.3, 40.7), (115, 0.7, 0.1, 41.2), (150, 0.6, 0.15, 41.3), (105, 1.0, 0.3, 41.7), (115, 0.9, 0.3, 41.9), (90, 1.0, 0.15, 42.4), (80, 1.1, 0.15, 42.6), (125, 0.8, 0.3, 42.9), (95, 0.9, 0.15, 44.0), (90, 1.0, 0.2, 44.6)]
  [(80, 1.1, 0.2, 45.5), (105, 0.8, 0.15, 45.7), (95, 0.9, 0.2, 48.1), (140, 0.7, 0.3, 48.9), (150, 0.5, 0.1, 49.5), (150, 0.6, 0.2, 49.8), (95, 1.1, 0.5, 4

In [10]:
def curve_arrays(key):
    curve = onion_by_key[key]['onion_layer_curves']
    ds = sorted(float(d) for d in curve)
    means = np.array([curve[str(d) if str(d) in curve else d]['mean'] for d in ds])
    return np.array(ds), means

for gi, group in enumerate(groups):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    for key in group:
        ds, means = curve_arrays(key)
        label = f"N={key[0]}, a={key[1]}, d={key[2]}"
        ax1.plot(ds, means, 'o-', label=label)
        ax2.plot(ds, means / means[-1], 'o-', label=label)  # normalized by largest-r value

    ax1.set_yscale('log')
    ax1.set_xlabel('Distance from reference node')
    ax1.set_ylabel('Mean R_eff,ij (Ohm, log scale)')
    ax1.set_title(f'Group {gi}: raw curves (matched scalar R_eff)')
    ax1.legend(fontsize=7)
    ax1.grid(True, alpha=0.3, which='both')

    ax2.set_xlabel('Distance from reference node')
    ax2.set_ylabel('R_eff,ij / R_eff,ij(r_max)')
    ax2.set_title(f'Group {gi}: shape (normalized by largest-r value)')
    ax2.legend(fontsize=7)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    exp_manager.save_figure(fig, f'partC_group_{gi}_shape_comparison.png')

/var/folders/2c/813sv0rj0qx09yx5yjmjvln40000gn/T/ipykernel_41168/3845552879.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  [OK] Figure saved: partC_group_0_shape_comparison.png


  [OK] Figure saved: partC_group_1_shape_comparison.png


  [OK] Figure saved: partC_group_2_shape_comparison.png


  [OK] Figure saved: partC_group_3_shape_comparison.png


...

  [OK] Figure saved: partC_group_24_shape_comparison.png
  [OK] Figure saved: partC_group_25_shape_comparison.png


  [OK] Figure saved: partC_group_26_shape_comparison.png


  [OK] Figure saved: partC_group_27_shape_comparison.png
  [OK] Figure saved: partC_group_28_shape_comparison.png


  [OK] Figure saved: partC_group_29_shape_comparison.png
  [OK] Figure saved: partC_group_30_shape_comparison.png


  [OK] Figure saved: partC_group_31_shape_comparison.png


In [11]:
# Thesis figures: the two highlighted groups (the strongest-signal group and the
# most shape-divergent one on the clipped-rule grid) saved again as SEPARATE,
# titleless panels. The combined figures above keep their axes titles for reading
# here in the notebook; in the dissertation each panel becomes a subfigure and the
# LaTeX subcaption carries that text instead. Same curves and same plotting order,
# so every triple keeps the color it has in the combined figure.
THESIS_GROUPS = (7, 31)

for gi in THESIS_GROUPS:
    for kind in ("raw", "shape"):
        fig, ax = plt.subplots(figsize=(6, 4.5))
        for key in groups[gi]:
            ds, means = curve_arrays(key)
            ax.plot(ds, means if kind == "raw" else means / means[-1], 'o-',
                    label=f"N={key[0]}, a={key[1]}, d={key[2]}")
        ax.set_xlabel('Distance from reference node')
        if kind == "raw":
            ax.set_yscale('log')
            ax.set_ylabel('Mean R_eff,ij (Ohm, log scale)')
            ax.grid(True, alpha=0.3, which='both')
        else:
            ax.set_ylabel('R_eff,ij / R_eff,ij(r_max)')
            ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7)
        plt.tight_layout()
        exp_manager.save_figure(fig, f'partC_group_{gi}_{kind}.png')
        plt.close(fig)


  [OK] Figure saved: partC_group_7_raw.png
  [OK] Figure saved: partC_group_7_shape.png
  [OK] Figure saved: partC_group_31_raw.png
  [OK] Figure saved: partC_group_31_shape.png


### C.1 A simple shape-divergence number per group

For each group, normalize every curve by its own value at the largest shared distance, then take
the RMS spread across curves at each shared layer distance, averaged over layers. Near 0 means
the curves are just rescaled copies of each other (scalar R_eff already captures everything the
onion curve does); larger values mean the shapes genuinely differ, i.e. the onion curve carries
information the scalar collapses away.

In [12]:
shape_divergence = []
for gi, group in enumerate(groups):
    curves = {key: curve_arrays(key) for key in group}
    shared_ds = sorted(set.intersection(*[set(ds) for ds, _ in curves.values()]))
    if len(shared_ds) < 2:
        continue

    normalized = []
    for key, (ds, means) in curves.items():
        idx = [list(ds).index(d) for d in shared_ds]
        ref_val = means[idx[-1]]
        normalized.append(means[idx] / ref_val)
    normalized = np.array(normalized)  # (n_curves, n_shared_layers)

    spread_per_layer = normalized.std(axis=0)
    divergence = float(spread_per_layer.mean())
    shape_divergence.append({'group': gi, 'keys': group, 'shared_layers': shared_ds, 'divergence': divergence})
    print(f"Group {gi} ({len(group)} triples, {len(shared_ds)} shared layers): "
          f"mean normalized-shape spread = {divergence:.4f}")

if shape_divergence:
    overall = np.mean([g['divergence'] for g in shape_divergence])
    print(f"\nOverall mean shape divergence across {len(shape_divergence)} group(s): {overall:.4f}")
else:
    print("No groups with >=2 shared layers yet -- nothing to score.")

Group 0 (5 triples, 9 shared layers): mean normalized-shape spread = 0.0088
Group 1 (8 triples, 9 shared layers): mean normalized-shape spread = 0.0184
Group 2 (4 triples, 9 shared layers): mean normalized-shape spread = 0.0347
Group 3 (13 triples, 9 shared layers): mean normalized-shape spread = 0.0287
Group 4 (10 triples, 9 shared layers): mean normalized-shape spread = 0.0296
Group 5 (12 triples, 9 shared layers): mean normalized-shape spread = 0.0356
Group 6 (6 triples, 9 shared layers): mean normalized-shape spread = 0.0352
Group 7 (14 triples, 9 shared layers): mean normalized-shape spread = 0.0506
Group 8 (9 triples, 9 shared layers): mean normalized-shape spread = 0.0329
Group 9 (14 triples, 9 shared layers): mean normalized-shape spread = 0.0407
Group 10 (11 triples, 9 shared layers): mean normalized-shape spread = 0.0601
Group 11 (10 triples, 9 shared layers): mean normalized-shape spread = 0.0463
Group 12 (10 triples, 9 shared layers): mean normalized-shape spread = 0.0394
G

---
# Part D -- Boosted rerun: does more statistical power confirm the signal?

Section C.2 found real but partial evidence at `N_TRIALS=6` (18% of tests significant vs. ~5%
expected by chance). `cluster/run_onion_grid_boost_task.py` reran just the `TOP_N_GROUPS=5`
strongest-signal groups (69 unique (N,a,d) triples) at `N_TRIALS_BOOST=50`, using the *same*
`stable_seed()` per combo -- so the first 6 realizations in the boosted data are byte-identical
to what's already in `cluster/onion_grid_results/` (verified directly: max abs diff 0.0 on a
sample combo), meaning this is a true extension of the existing draws, not an independent
resample.

In [13]:
if not RUN_BOOST:
    print('Part D skipped (RUN_BOOST=False): no boost run for this grid')
else:
    from scipy import stats

    boost_dir = os.path.join(cluster_dir, "onion_grid_boost_results")
    boost_plan_path = os.path.join(cluster_dir, "onion_grid_boost_plan.json")

    boost_rows, n_boost_found, _ = load_chunks(boost_dir)
    boost_by_key = {(r['N'], r['a'], r['d']): r for r in boost_rows}

    with open(boost_plan_path) as f:
        boost_plan = json.load(f)
    selected_groups = boost_plan['selected_groups']
    n_trials_boost = boost_plan['config']['N_TRIALS_BOOST']

    print(f"Loaded {len(boost_by_key)} boosted combos across {len(selected_groups)} groups "
          f"(target was {len(set(tuple(t) for g in selected_groups for t in g['triples']))} unique triples)")

    def per_realization_normalized_boost(key, shared_layers):
        out = _aligned_curves(boost_by_key[key], shared_layers)
        if len(out) == 0:
            return out
        return out / out[:, -1:]

    boost_all_pvals = []
    boost_group_results = []
    for g in selected_groups:
        group = [tuple(t) for t in g['triples']]
        if not all(k in boost_by_key for k in group):
            print(f"  [SKIP] group {g['group_index']}: boosted data incomplete")
            continue

        curve_ds = {key: sorted(float(d) for d in boost_by_key[key]['onion_layer_curves']) for key in group}
        shared = sorted(set.intersection(*[set(ds) for ds in curve_ds.values()]))
        per_key_curves = {key: per_realization_normalized_boost(key, shared) for key in group}
        per_key_curves = {k: v for k, v in per_key_curves.items() if v.shape[0] >= 3}

        layer_pvals = []
        for li, d in enumerate(shared):
            if li == len(shared) - 1:
                continue
            samples = [c[:, li] for c in per_key_curves.values()]
            _, p_val = stats.f_oneway(*samples)
            layer_pvals.append(p_val)
            boost_all_pvals.append(p_val)

        n_sig = sum(1 for p in layer_pvals if p < 0.05)
        boost_group_results.append({'group': g['group_index'], 'min_p_before': g['min_p'],
                                    'min_p_after': float(min(layer_pvals)),
                                    'frac_sig_before': g['frac_sig_p05'],
                                    'frac_sig_after': n_sig / len(layer_pvals)})
        print(f"group {g['group_index']:2d}: min_p {g['min_p']:.2e} -> {min(layer_pvals):.2e}   "
              f"(N_TRIALS=6 -> {n_trials_boost}),  layers p<0.05: {g['frac_sig_p05']:.0%} -> {n_sig}/{len(layer_pvals)} ({n_sig/len(layer_pvals):.0%})")

    n_boost_tests = len(boost_all_pvals)
    n_boost_sig = sum(1 for p in boost_all_pvals if p < 0.05)
    boost_bonf_alpha = 0.05 / n_boost_tests
    n_boost_bonf = sum(1 for p in boost_all_pvals if p < boost_bonf_alpha)

    print(f"\nTOTAL (boosted): {n_boost_sig}/{n_boost_tests} significant at p<0.05 ({n_boost_sig/n_boost_tests:.0%}), "
          f"{n_boost_bonf} survive Bonferroni correction (alpha={boost_bonf_alpha:.2e})")
    print(f"(For comparison, the original N_TRIALS=6 run across all 214 tests: 39/214 significant (18%), 2 survived Bonferroni)")

Part D skipped (RUN_BOOST=False): no boost run for this grid


In [14]:
if not RUN_BOOST:
    print('Part D skipped (RUN_BOOST=False): no boost run for this grid')
else:
    def boost_curve_arrays(key):
        curve = boost_by_key[key]['onion_layer_curves']
        ds = sorted(float(d) for d in curve)
        means = np.array([curve[str(d)]['mean'] for d in ds])
        return np.array(ds), means

    strongest = min(selected_groups, key=lambda g: g['min_p'])
    group = [tuple(t) for t in strongest['triples']]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    for key in group:
        ds, means = boost_curve_arrays(key)
        label = f"N={key[0]}, a={key[1]}, d={key[2]}"
        ax1.plot(ds, means, 'o-', label=label, markersize=3)
        ax2.plot(ds, means / means[-1], 'o-', label=label, markersize=3)

    ax1.set_yscale('log')
    ax1.set_xlabel('Distance from reference node')
    ax1.set_ylabel('Mean R_eff,ij (Ohm, log scale)')
    ax1.set_title(f"Group {strongest['group_index']} at N_TRIALS={n_trials_boost}: raw curves")
    ax1.legend(fontsize=6)
    ax1.grid(True, alpha=0.3, which='both')

    ax2.set_xlabel('Distance from reference node')
    ax2.set_ylabel('R_eff,ij / R_eff,ij(r_max)')
    ax2.set_title('Shape (normalized) -- distinctly separated, not just noisy overlap')
    ax2.legend(fontsize=6)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    exp_manager.save_figure(fig, f"partD_group_{strongest['group_index']}_boosted_shape.png")

Part D skipped (RUN_BOOST=False): no boost run for this grid


### C.2 Is that divergence bigger than noise, or smaller? (proper test)

**Earlier version of this section was wrong and has been corrected.** It compared the spread of
group *means* (each mean already averaged over `N_TRIALS` realizations) against the *raw*
single-realization spread within a triple -- not apples-to-apples, since averaging over N
realizations should shrink sampling noise by ~sqrt(N). That flawed comparison undercounted the
signal and produced a false negative.

The correct test: for each shared layer within a matched-R_eff group, treat "triple" as the
group variable and "realization" as the replicate (one-way ANOVA, `scipy.stats.f_oneway`) -- is
the variance *between* triples' individual realization curves larger than the variance *within*
each triple, more than expected by chance given the actual `n` per triple? This uses the
individual per-realization curves directly (`per_realization_means`), not the pre-averaged means,
so no information is thrown away and no sqrt(N) correction has to be done by hand. The reference
layer (every curve is exactly 1.0 there by construction, from the normalization) is excluded --
it has zero variance in every group and isn't a meaningful test.

In [15]:
from scipy import stats

def per_realization_normalized(key, shared_layers):
    """Individual (not averaged) realization curves for `key`, restricted to
    shared_layers, each normalized by its OWN value at the largest shared layer."""
    out = _aligned_curves(onion_by_key[key], shared_layers)
    if len(out) == 0:
        return out
    return out / out[:, -1:]

anova_results = []
all_pvals = []
for gi, group in enumerate(groups):
    curve_ds = {key: sorted(float(d) for d in onion_by_key[key]['onion_layer_curves']) for key in group}
    shared = sorted(set.intersection(*[set(ds) for ds in curve_ds.values()]))
    if len(shared) < 2:
        continue

    per_key_curves = {key: per_realization_normalized(key, shared) for key in group}
    per_key_curves = {k: v for k, v in per_key_curves.items() if v.shape[0] >= 3}  # need >=3 reps for a real test
    if len(per_key_curves) < 2:
        continue

    layer_pvals = []
    for li, d in enumerate(shared):
        if li == len(shared) - 1:
            continue  # reference layer: exactly 1.0 for every curve by construction
        samples = [curves[:, li] for curves in per_key_curves.values()]
        _, p_val = stats.f_oneway(*samples)
        layer_pvals.append(p_val)
        all_pvals.append(p_val)

    if layer_pvals:
        anova_results.append({'group': gi, 'n_triples': len(per_key_curves), 'n_layers_tested': len(layer_pvals),
                              'min_p': float(min(layer_pvals)), 'frac_sig_p05': float(np.mean([p < 0.05 for p in layer_pvals]))})

n_tests = len(all_pvals)
n_sig_005 = sum(1 for p in all_pvals if p < 0.05)
bonferroni_alpha = 0.05 / n_tests if n_tests else float('nan')
n_sig_bonf = sum(1 for p in all_pvals if p < bonferroni_alpha)

print(f"{n_tests} (group, layer) ANOVA tests across {len(anova_results)} groups")
print(f"Significant at p<0.05 (uncorrected): {n_sig_005}/{n_tests} ({n_sig_005/n_tests:.1%}) "
      f"-- ~5% expected under the null of no real shape difference")
print(f"Significant at Bonferroni-corrected alpha={bonferroni_alpha:.2e}: {n_sig_bonf}/{n_tests}")
print(f"Median p-value: {np.median(all_pvals):.4f}   Mean p-value: {np.mean(all_pvals):.4f}   "
      f"(0.5 expected on average under a uniform null)")

print("\nStrongest-signal groups (lowest min p-value):")
for gr in sorted(anova_results, key=lambda g: g['min_p'])[:8]:
    print(f"  group {gr['group']:2d}: {gr['n_triples']} triples, min_p={gr['min_p']:.2e}, "
          f"frac layers p<0.05 = {gr['frac_sig_p05']:.0%}")

if n_sig_005 / n_tests > 0.10:
    print(f"\n=> {n_sig_005/n_tests:.0%} of tests significant is well above the ~5% expected by chance, "
          f"and {n_sig_bonf} test(s) survive strict Bonferroni correction across all {n_tests} tests. "
          f"There IS detectable evidence that onion-curve shape differs between (N,a,d) triples with "
          f"matched scalar R_eff -- but it's a partial/noisy signal at the base realization count, not uniformly present "
          f"across all matched groups (most groups show no significant layers).")

234 (group, layer) ANOVA tests across 32 groups
Significant at p<0.05 (uncorrected): 81/234 (34.6%) -- ~5% expected under the null of no real shape difference
Significant at Bonferroni-corrected alpha=2.14e-04: 19/234
Median p-value: 0.1758   Mean p-value: 0.2587   (0.5 expected on average under a uniform null)

Strongest-signal groups (lowest min p-value):
  group  7: 14 triples, min_p=9.27e-09, frac layers p<0.05 = 88%
  group  5: 12 triples, min_p=1.73e-08, frac layers p<0.05 = 50%
  group  3: 13 triples, min_p=1.49e-06, frac layers p<0.05 = 62%
  group 10: 11 triples, min_p=1.86e-06, frac layers p<0.05 = 75%
  group  6: 6 triples, min_p=3.89e-06, frac layers p<0.05 = 62%
  group  2: 4 triples, min_p=6.66e-06, frac layers p<0.05 = 88%
  group 25: 11 triples, min_p=1.24e-05, frac layers p<0.05 = 62%
  group  4: 10 triples, min_p=1.32e-04, frac layers p<0.05 = 38%

=> 35% of tests significant is well above the ~5% expected by chance, and 19 test(s) survive strict Bonferroni correction

---
# Part D -- Boosted rerun of the strongest-signal groups (N_TRIALS=50)

Section C.2 found real but partial evidence at `N_TRIALS=6`: 18% of 214 (group, layer) ANOVA
tests significant at p<0.05 (vs ~5% expected by chance), only 2 surviving Bonferroni correction.
Most of that came from a handful of groups (6, 9, 11, 16, 4) with the strongest signal already at
N_TRIALS=6. `cluster/run_onion_grid_boost_task.py` reran exactly those 69 combos (the union of
those 5 groups' triples) at `N_TRIALS=50` -- same `stable_seed()`, so the first 6 realizations
per combo are identical to what's already in `onion_grid_results/`, this is a strict extension,
not an independent resample. This section redoes the same per-(group, layer) ANOVA from Section
C.2 on the boosted data, to see whether more samples turn the noisy partial signal into something
thesis-defensible.

### D.1 Load the boosted data and re-run the ANOVA

In [16]:
if not RUN_BOOST:
    print('Part D skipped (RUN_BOOST=False): no boost run for this grid')
else:
    boost_dir = os.path.join(cluster_dir, "onion_grid_boost_results")
    N_TASKS_BOOST = 20

    boost_rows, n_boost_found, n_boost_tasks = load_chunks(boost_dir, N_TASKS_BOOST)
    onion_boost_by_key = {(r['N'], r['a'], r['d']): r for r in boost_rows}
    print(f"Boosted onion grid: {n_boost_found}/{n_boost_tasks} chunk files found, "
          f"{len(onion_boost_by_key)} combos loaded (N_TRIALS=50 each)")

    with open(os.path.join(cluster_dir, "onion_grid_boost_plan.json")) as f:
        boost_plan = json.load(f)
    boosted_group_indices = {g['group_index'] for g in boost_plan['selected_groups']}
    print(f"Boosted groups (by Part C.2 numbering): {sorted(boosted_group_indices)}")


Part D skipped (RUN_BOOST=False): no boost run for this grid


In [17]:
if not RUN_BOOST:
    print('Part D skipped (RUN_BOOST=False): no boost run for this grid')
else:
    def per_realization_normalized_from(source_by_key, key, shared_layers):
        """Same as Section C.2's per_realization_normalized, parameterized by which
        dict (original onion_by_key vs onion_boost_by_key) to pull realizations from."""
        out = _aligned_curves(source_by_key[key], shared_layers)
        if len(out) == 0:
            return out
        return out / out[:, -1:]


    def anova_for_group(gi, source_by_key, min_reps=3):
        group = groups[gi]
        curve_ds = {key: sorted(float(d) for d in source_by_key[key]['onion_layer_curves'])
                   for key in group if key in source_by_key}
        shared = sorted(set.intersection(*[set(ds) for ds in curve_ds.values()])) if curve_ds else []
        if len(shared) < 2:
            return None

        per_key_curves = {key: per_realization_normalized_from(source_by_key, key, shared) for key in curve_ds}
        per_key_curves = {k: v for k, v in per_key_curves.items() if v.shape[0] >= min_reps}
        if len(per_key_curves) < 2:
            return None

        layer_pvals = []
        for li, d in enumerate(shared):
            if li == len(shared) - 1:
                continue
            samples = [curves[:, li] for curves in per_key_curves.values()]
            _, p_val = stats.f_oneway(*samples)
            layer_pvals.append(p_val)

        if not layer_pvals:
            return None
        return {'n_triples': len(per_key_curves), 'n_layers_tested': len(layer_pvals),
                'min_p': float(min(layer_pvals)), 'frac_sig_p05': float(np.mean([p < 0.05 for p in layer_pvals])),
                'all_p': layer_pvals}


    comparison = []
    for gi in sorted(boosted_group_indices):
        before = anova_for_group(gi, onion_by_key)
        after = anova_for_group(gi, onion_boost_by_key)
        comparison.append({'group': gi, 'before': before, 'after': after})
        b_str = f"min_p={before['min_p']:.2e}, {before['frac_sig_p05']:.0%} sig" if before else "n/a"
        a_str = f"min_p={after['min_p']:.2e}, {after['frac_sig_p05']:.0%} sig" if after else "n/a"
        print(f"group {gi:2d}:  N=6  {b_str:32s} ->  N=50  {a_str}")

    boost_all_p = [p for c in comparison for p in (c['after']['all_p'] if c['after'] else [])]
    n_boost_tests = len(boost_all_p)
    n_boost_sig = sum(1 for p in boost_all_p if p < 0.05)
    bonf_alpha_boost = 0.05 / n_boost_tests if n_boost_tests else float('nan')
    n_boost_bonf = sum(1 for p in boost_all_p if p < bonf_alpha_boost)
    print(f"\nBoosted groups only, N_TRIALS=50: {n_boost_sig}/{n_boost_tests} tests significant at "
          f"p<0.05 ({n_boost_sig/n_boost_tests:.1%}), {n_boost_bonf} survive Bonferroni "
          f"(alpha={bonf_alpha_boost:.2e})")


Part D skipped (RUN_BOOST=False): no boost run for this grid


### D.2 Before vs after, per group

In [18]:
if not RUN_BOOST:
    print('Part D skipped (RUN_BOOST=False): no boost run for this grid')
else:
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    gs = [c['group'] for c in comparison]
    before_p = [c['before']['min_p'] if c['before'] else np.nan for c in comparison]
    after_p = [c['after']['min_p'] if c['after'] else np.nan for c in comparison]

    x = np.arange(len(gs))
    width = 0.35
    ax.bar(x - width/2, before_p, width, label='N_TRIALS=6 (original)', color='steelblue')
    ax.bar(x + width/2, after_p, width, label='N_TRIALS=50 (boosted)', color='crimson')
    ax.set_yscale('log')
    ax.axhline(0.05, color='gray', linestyle='--', linewidth=1, label='p=0.05')
    ax.axhline(bonf_alpha_boost, color='black', linestyle=':', linewidth=1,
              label=f'Bonferroni alpha={bonf_alpha_boost:.1e}')
    ax.set_xticks(x)
    ax.set_xticklabels([f'group {g}' for g in gs])
    ax.set_ylabel('min p-value across layers (log scale)')
    ax.set_title('ANOVA min p-value per group: before vs after boosting N_TRIALS')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()
    exp_manager.save_figure(fig, "partD_boost_pvalue_comparison.png")


Part D skipped (RUN_BOOST=False): no boost run for this grid


---
## Part D notes

Boosting the 5 strongest-signal groups from `N_TRIALS=6` to `N_TRIALS=50` (69 combos, same
`stable_seed()`-derived draws extended, not resampled) sharpens the ANOVA result exactly where it
already mattered most. Compare the "boosted groups only" statistics here against Section C.2's
full-grid numbers (39/214 = 18.2% significant, 2/214 survive Bonferroni at N_TRIALS=6) -- if the
signal in these 5 groups is real, more realizations should shrink the per-triple sampling noise
and push more of their layers under the significance threshold, since a one-way ANOVA's power
scales with sample size for a fixed true effect size. If it does NOT sharpen substantially, that
argues the original min_p values in these groups were themselves partly noise-driven lucky draws
rather than a robust effect -- worth reporting honestly either way, since a boost that fails to
replicate is itself informative (guards against p-hacking by selecting-then-not-confirming).

### D.3 Save

In [19]:
if not RUN_BOOST:
    print('Part D skipped (RUN_BOOST=False): no boost run for this grid')
else:
    boost_summary = {
        'boost_config': boost_plan['config'],
        'boosted_group_comparison': [
            {'group': c['group'],
             'before': {k: v for k, v in c['before'].items() if k != 'all_p'} if c['before'] else None,
             'after': {k: v for k, v in c['after'].items() if k != 'all_p'} if c['after'] else None}
            for c in comparison
        ],
        'boosted_only_test_summary': {
            'n_tests': n_boost_tests, 'n_sig_p05': n_boost_sig,
            'frac_sig_p05': n_boost_sig / n_boost_tests if n_boost_tests else None,
            'bonferroni_alpha': bonf_alpha_boost, 'n_sig_bonferroni': n_boost_bonf,
        },
    }
    exp_manager.save_json(boost_summary, "r_eff_ij_boost_summary.json")
    print("Saved Part D boost summary.")


Part D skipped (RUN_BOOST=False): no boost run for this grid


---
## Notes

- This notebook's networks are **paired** with the scalar R_eff grid in
  `effective_resistance_vs_d.ipynb` Part C by construction: `cluster/run_onion_grid_task.py` uses
  the exact same grid, `stable_seed()` formula, and network-draw order as
  `cluster/run_effective_resistance_d_task.py`, so identical (N, a, d) combos produce identical
  networks in both. This only holds for onion-grid data computed *after* the seed-hashing fix
  (`utils/pipeline_utils.py`'s `stable_seed()`, replacing the old `hash((..., 'ext'))` formula,
  which was randomized per-process and could not actually reproduce networks across separate runs
  -- see that function's docstring). The same property extends to the Part D boosted rerun
  (`run_onion_grid_boost_task.py`): its first 6 realizations per combo are byte-identical to the
  original `N_TRIALS=6` data (verified directly, max abs diff 0.0), so it's a strict extension of
  the same draws, not an independent resample.
- Part C's grouping (`REL_TOL = 0.15`, requiring different (N, a)) is a starting cut, not a fixed
  methodology -- worth revisiting once there's enough data to be choosier about what counts as
  "matched" scalar R_eff.
- **Result on the full 420-combo grid (2026-08-24, `N_TRIALS=6`):** an earlier version of Section
  C.2 compared the cross-triple spread of group *mean* curves against the *raw* single-realization
  noise within a triple, without correcting for the ~sqrt(N_TRIALS) reduction in noise averaging
  provides -- that comparison was wrong and produced a false "no signal" conclusion. The corrected
  Section C.2 runs a proper one-way ANOVA per (group, layer) directly on the individual
  realization curves: 18% of 214 tests significant at p<0.05 (vs. ~5% expected by chance), 2
  survive strict Bonferroni correction -- real but partial evidence, most matched groups not yet
  significant.
- **Result after boosting the 5 strongest groups to `N_TRIALS=50` (Part D, 2026-08-24):** the
  signal sharpens dramatically and confirms it's real, not a fluke of the smaller sample.
  **35/40 tests significant at p<0.05 (88%, up from 18%), and 31/40 survive strict Bonferroni
  correction (up from 2/214).** Four of the five groups reach 8/8 significant layers with p-values
  down to ~1e-48; even the weakest of the five reaches 3/8 significant layers at p=2.3e-5.
  **Conclusion: there is strong, statistically robust evidence that R_eff,ij(r) onion-curve shape
  carries real information about (N, a, d) beyond what the scalar source-drain R_eff captures** --
  triples that are indistinguishable by scalar R_eff alone (within 15%) are, in at least these
  five regions of parameter space, clearly distinguishable by onion-curve shape. This directly
  answers the original question this notebook was built to investigate.
- A follow-on curve-based (N,a,d) inversion notebook -- the 3-parameter, curve-shape analogue of
  `inverse_N_a_from_R_eff.ipynb`, actually predicting (N,a,d) from a measured onion curve rather
  than just testing whether curves differ -- is the natural next step and is now well-motivated by
  this result.

---
## Save summary

In [20]:
summary = {
    'experiment': exp_manager.get_info(),
    'part_a_preview': [{'N': r['N'], 'a': r['a'], 'd': r['d'], 'n': r['n'], 'mean_ohm': r['mean_ohm']}
                       for r in preview_rows],
    'grid_coverage': {'onion_chunks_found': n_onion_found, 'scalar_chunks_found': n_scalar_found,
                      'n_tasks': n_tasks, 'combos_loaded_onion': len(onion_by_key),
                      'combos_loaded_scalar': len(scalar_by_key)},
    'shape_divergence_groups': shape_divergence,
    'anova_test': {
        'n_tests': n_tests,
        'n_sig_p05': n_sig_005,
        'frac_sig_p05': n_sig_005 / n_tests if n_tests else None,
        'bonferroni_alpha': bonferroni_alpha,
        'n_sig_bonferroni': n_sig_bonf,
        'median_p': float(np.median(all_pvals)) if all_pvals else None,
        'mean_p': float(np.mean(all_pvals)) if all_pvals else None,
        'per_group': anova_results,
    },
    'boosted_anova_test': ({
        'n_trials_boost': n_trials_boost,
        'n_boost_combos_found': len(boost_by_key),
        'n_tests': n_boost_tests,
        'n_sig_p05': n_boost_sig,
        'frac_sig_p05': n_boost_sig / n_boost_tests if n_boost_tests else None,
        'bonferroni_alpha': boost_bonf_alpha,
        'n_sig_bonferroni': n_boost_bonf,
        'per_group': boost_group_results,
    } if RUN_BOOST else None),
    'boundary': BOUNDARY,
}
exp_manager.save_json(summary, 'r_eff_ij_vs_N_a_d_summary.json')

print(f"\nAll results saved to: {exp_manager.experiment_dir}")
for f in sorted(os.listdir(exp_manager.experiment_dir)):
    print(f"  - {f}")

  [OK] JSON saved: r_eff_ij_vs_N_a_d_summary.json

All results saved to: /Users/jefferson/Library/CloudStorage/GoogleDrive-jclc@cin.ufpe.br/.shortcut-targets-by-id/1rYn0mY-dmEuA4NKcrSO1-KsKjMo6XXH_/20242_PiaNo/ColabProject/utils/../experiments/exp_877
  - partA_preview_onion_curves.png
  - partC_group_0_shape_comparison.png
  - partC_group_10_shape_comparison.png
  - partC_group_11_shape_comparison.png
  - partC_group_12_shape_comparison.png
  - partC_group_13_shape_comparison.png
  - partC_group_14_shape_comparison.png
  - partC_group_15_shape_comparison.png
  - partC_group_16_shape_comparison.png
  - partC_group_17_shape_comparison.png
  - partC_group_18_shape_comparison.png
  - partC_group_19_shape_comparison.png
  - partC_group_1_shape_comparison.png
  - partC_group_20_shape_comparison.png
  - partC_group_21_shape_comparison.png
  - partC_group_22_shape_comparison.png
  - partC_group_23_shape_comparison.png
  - partC_group_24_shape_comparison.png
  - partC_group_25_shape_comparison